# DeFoG — Marginal Ablation (QM9)

Runs two training + testing experiments back-to-back to isolate the effect
of the initial noise distribution on generation quality:

| Run | Transition | Marginals source |
|-----|-----------|------------------|
| `qm9_ablation_marginal` | `marginal` | Dataset type-frequency statistics |
| `qm9_ablation_spminer` | `loaded_marginal` | SPMiner-mined motif marginals |

**Prerequisites:** `02_session_init.ipynb` must be completed in this session.

Each run: 100 epochs training → test with last checkpoint → results logged to W&B.  
Checkpoints are saved directly to Google Drive via the symlink set up in session init.

In [ ]:
# ── Cell 1: Paths, environment, and helper functions ──────────────────────────
import os
import glob
import subprocess
import sys

# ── Paths ─────────────────────────────────────────────────────────────────────
PYTHON     = '/content/miniconda3/envs/defog/bin/python'
REPO_DIR   = '/content/DeFoGPlus'
SRC        = f'{REPO_DIR}/src'
DATA_QM9   = f'{REPO_DIR}/data/qm9'
DRIVE_BASE = '/content/drive/MyDrive/DeFoGColab'

# Passed to every subprocess so graph_tool does not attempt to open a display.
RUN_ENV = {**os.environ, 'MPLBACKEND': 'Agg'}

# ── Shared Hydra overrides for both runs ──────────────────────────────────────
# check_val_every_n_epochs=20  → validate at epochs 20, 40, 60, 80, 100
# sample_every_val=1           → checkpoint saved at every validation
# samples_to_generate=128      → small mid-training sample; keeps val fast
# final_model_samples_to_generate=512 → test budget
COMMON = [
    '+experiment=qm9_no_h',
    'hydra.job.chdir=False',
    'train.n_epochs=100',
    'general.check_val_every_n_epochs=20',
    'general.sample_every_val=1',
    'general.samples_to_generate=128',
    'general.final_model_samples_to_generate=512',
]


# ── Streaming subprocess helper ───────────────────────────────────────────────
def run_live(cmd: list, cwd: str = SRC):
    """Run cmd, streaming stdout+stderr to the cell output line by line.
    Raises RuntimeError if the process exits with a non-zero code.
    """
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=cwd,
        env=RUN_ENV,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Process exited with code {proc.returncode}')


# ── Train-and-test helper ─────────────────────────────────────────────────────
def train_and_test(name: str, extra: list):
    """Train for 100 epochs then immediately test with the last checkpoint.

    Args:
        name:  W&B run name and checkpoint subdirectory name.
        extra: Additional Hydra overrides specific to this run
               (e.g. model.transition=..., model.motif_marginals_path=...).
    """
    base_args = COMMON + [f'general.name={name}'] + extra

    # ── Training ──────────────────────────────────────────────────────────────
    print(f'\n{"╔" + "═"*58 + "╗"}')
    print(f'  TRAIN  →  {name}')
    print(f'{"╚" + "═"*58 + "╝"}\n')

    run_live([PYTHON, 'main.py'] + base_args)

    # ── Locate last checkpoint ─────────────────────────────────────────────
    # src/checkpoints/ is symlinked to Drive, so this path is persistent.
    ckpt_dir = f'{SRC}/checkpoints/{name}'
    ckpts = sorted(
        glob.glob(f'{ckpt_dir}/*.ckpt'),
        key=os.path.getmtime
    )
    if not ckpts:
        raise RuntimeError(
            f'No checkpoint found in {ckpt_dir}.\n'
            f'Check that train.save_model is True and the symlink is correct.'
        )
    ckpt = ckpts[-1]
    print(f'\nLast checkpoint: {ckpt}')

    # ── Testing ────────────────────────────────────────────────────────────
    print(f'\n{"╔" + "═"*58 + "╗"}')
    print(f'  TEST   →  {name}')
    print(f'{"╚" + "═"*58 + "╝"}\n')

    run_live([PYTHON, 'main.py'] + base_args + [f'general.test_only={ckpt}'])

    print(f'\n✓  {name} complete.  Checkpoint: {ckpt}')
    return ckpt


print('Helpers loaded.')
print(f'SRC      : {SRC}')
print(f'DATA_QM9 : {DATA_QM9}')

In [ ]:
# ── Cell 2: Pre-flight checks ─────────────────────────────────────────────────
import torch

errors = []

# GPU
if torch.cuda.is_available():
    print(f'✓  GPU: {torch.cuda.get_device_name(0)}')
else:
    errors.append('No CUDA GPU detected — training will be extremely slow.')

# SPMiner marginals .pt file
spminer_pt = f'{DATA_QM9}/spminer_motif_marginals.pt'
if os.path.isfile(spminer_pt):
    m = torch.load(spminer_pt, map_location='cpu', weights_only=True)
    print(f'✓  SPMiner marginals: X={list(m["X"].shape)}  E={list(m["E"].shape)}')
else:
    errors.append(f'Missing: {spminer_pt}')

# Checkpoint symlink
ckpt_link = f'{SRC}/checkpoints'
ckpt_target = f'{DRIVE_BASE}/checkpoints'
if os.path.islink(ckpt_link) and os.readlink(ckpt_link) == ckpt_target:
    print(f'✓  Checkpoint symlink → {ckpt_target}')
else:
    errors.append(f'Checkpoint symlink missing or wrong. Run 02_session_init.ipynb first.')

# W&B key
if os.environ.get('WANDB_API_KEY', ''):
    print('✓  WANDB_API_KEY is set.')
else:
    print('⚠  WANDB_API_KEY not set — runs will not be logged to W&B.')

if errors:
    for e in errors:
        print(f'✗  {e}')
    raise RuntimeError('Pre-flight failed. Fix the issues above before continuing.')
else:
    print('\nAll checks passed — ready to train.')

In [ ]:
# ── Cell 3: Run 1 — unedited marginal distribution ────────────────────────────
#
# Baseline: the initial noise z_T is sampled from the dataset's empirical
# type-frequency marginals (no motif editing of any kind).
# This is the standard DeFoG marginal transition.

ckpt_marginal = train_and_test(
    name  = 'qm9_ablation_marginal',
    extra = ['model.transition=marginal'],
)

In [ ]:
# ── Cell 4: Run 2 — SPMiner motif-edited marginal distribution ────────────────
#
# The initial noise z_T is sampled from marginals that were estimated from
# QM9 graphs after injecting the top-1 SPMiner-mined motif (a 3-node chain).
# The only change relative to Run 1 is the rate matrix; the model architecture,
# training loop, and graph size are all identical.

SPMINER_PT = f'{DATA_QM9}/spminer_motif_marginals.pt'

ckpt_spminer = train_and_test(
    name  = 'qm9_ablation_spminer',
    extra = [
        'model.transition=loaded_marginal',
        f'model.motif_marginals_path={SPMINER_PT}',
    ],
)

In [ ]:
# ── Cell 5: Summary ───────────────────────────────────────────────────────────
print('=' * 60)
print('  Ablation complete')
print('=' * 60)
print()
rows = [
    ('qm9_ablation_marginal', 'marginal',        ckpt_marginal),
    ('qm9_ablation_spminer',  'loaded_marginal', ckpt_spminer),
]
for run_name, transition, ckpt in rows:
    print(f'Run      : {run_name}')
    print(f'Transition: {transition}')
    print(f'Checkpoint: {ckpt}')
    print()

print('Results are logged to W&B under the run names above.')
print('Checkpoints are persisted at:')
print(f'  {DRIVE_BASE}/checkpoints/')